# CMIP6 core API example

Shows the direct public API with one built-in core fix.

Flow: check -> dry-run -> apply -> re-check.

In [ ]:
import numpy as np

import woodpecker
from woodpecker.testing import make_cmip6

Create a CMIP6-like dataset where `tas` is stored in Celsius instead of Kelvin.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)
original_values = dataset["tas"].values.copy()

dataset

In [ ]:
fix_ids = ["woodpecker.normalize_tas_units_to_kelvin"]

findings = woodpecker.check(dataset, fixes=fix_ids)
findings.fix_ids

Dry-run previews the repair without changing the dataset.

In [ ]:
result = woodpecker.apply(dataset, fixes=findings.fix_ids, dry_run=True)

(
    result.stats,
    result.preview,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values),
)

Apply the fix in memory and re-check.

In [ ]:
write = woodpecker.apply(dataset, fixes=findings.fix_ids, dry_run=False)

(
    write.stats,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values + 273.15),
)

In [ ]:
recheck = woodpecker.check(dataset, fixes=fix_ids)
bool(recheck)